# World Cup 2026 - Task 2.1: dataset processing
Turns the 3 raw files into the final 104-row dataset.
Inputs: the fbref schedule, the FIFA ranking and the historical results.
The regression is in a separate notebook.

## 1. Setup

In [ ]:
import re
from collections import defaultdict
from io import StringIO

import pandas as pd

# the three input files
SCHEDULE_FILE = "raw_fbref_schedule_export.csv"
RANKING_FILE = "raw_fifa_ranking_11june2026.csv"
HISTORY_FILE = "raw_international_results.csv"

## 2. Settings and lookups

In [ ]:
KICKOFF = "2026-06-11"   # only matches before this date count as history
FORM_WINDOW = 10         # "recent form" = last 10 matches

# tournament stage as a number
ROUND_NUM = {"Group stage": 1, "Round of 32": 2, "Round of 16": 3,
             "Quarter-finals": 4, "Semi-finals": 5, "Third-place match": 6, "Final": 6}

HOSTS = {"Mexico", "Canada", "USA"}

# typed in by hand, none of the raw files have confederations
CONFED = {
    "Mexico": "CONCACAF", "South Africa": "CAF", "Korea Republic": "AFC", "Czechia": "UEFA",
    "Canada": "CONCACAF", "Bosnia and Herzegovina": "UEFA", "USA": "CONCACAF", "Paraguay": "CONMEBOL",
    "Qatar": "AFC", "Switzerland": "UEFA", "Brazil": "CONMEBOL", "Morocco": "CAF",
    "Australia": "AFC", "Türkiye": "UEFA", "Haiti": "CONCACAF", "Scotland": "UEFA",
    "Germany": "UEFA", "Curaçao": "CONCACAF", "Netherlands": "UEFA", "Japan": "AFC",
    "Côte d'Ivoire": "CAF", "Ecuador": "CONMEBOL", "Sweden": "UEFA", "Tunisia": "CAF",
    "Belgium": "UEFA", "Egypt": "CAF", "Spain": "UEFA", "Cabo Verde": "CAF",
    "Saudi Arabia": "AFC", "Uruguay": "CONMEBOL", "IR Iran": "AFC", "New Zealand": "OFC",
    "France": "UEFA", "Senegal": "CAF", "Iraq": "AFC", "Norway": "UEFA", "Argentina": "CONMEBOL",
    "Algeria": "CAF", "Austria": "UEFA", "Jordan": "AFC", "Portugal": "UEFA", "Congo DR": "CAF",
    "England": "UEFA", "Croatia": "UEFA", "Ghana": "CAF", "Panama": "CONCACAF",
    "Uzbekistan": "AFC", "Colombia": "CONMEBOL",
}

# the sources spell some team names differently, so everything is converted to the FIFA spelling
FBREF_NAME_FIX = {"Bosnia–Herz": "Bosnia and Herzegovina"}
HISTORY_NAME_FIX = {
    "Czech Republic": "Czechia", "Turkey": "Türkiye", "Iran": "IR Iran",
    "Cape Verde": "Cabo Verde", "Ivory Coast": "Côte d'Ivoire", "DR Congo": "Congo DR",
    "South Korea": "Korea Republic", "United States": "USA",
}

## 3. Match schedule (fbref)

In [ ]:
def clean_fbref_team(raw):
    # fbref adds a small country code, e.g. "Mexico mx", which is removed here
    tokens = [t for t in str(raw).strip().split(" ") if not re.fullmatch(r"[a-z]{2,3}", t)]
    name = " ".join(tokens).strip()
    return FBREF_NAME_FIX.get(name, name)


def read_schedule(path):
    # reads the CSV, or the .xls that fbref exports (which is really an HTML table)
    if path.lower().endswith((".xls", ".html", ".htm")):
        with open(path, encoding="utf-8") as f:
            df = pd.read_html(StringIO(f.read()))[0]
    else:
        df = pd.read_csv(path, encoding="utf-8")
    return df.dropna(subset=["Round"])  # removes the blank separator rows


matches = []
for _, row in read_schedule(SCHEDULE_FILE).iterrows():
    score = str(row["Score"]).strip()
    core = re.sub(r"\(\d+\)", "", score).strip()  # removes the penalty shoot-out numbers
    goals1, goals2 = [int(x) for x in re.split(r"[–-]", core)]
    matches.append({
        "date": str(row["Date"]).strip(),
        "round_number": ROUND_NUM[row["Round"].strip()],
        "team1": clean_fbref_team(row["Home"]),
        "team2": clean_fbref_team(row["Away"]),
        "goals1": goals1,
        "goals2": goals2,
    })

print("Matches loaded:", len(matches))
assert len(matches) == 104

## 4. FIFA ranking points (11 June 2026)

In [ ]:
rank_df = pd.read_csv(RANKING_FILE, encoding="utf-8")
fifa_points = dict(zip(rank_df["team"].str.strip(), rank_df["points"].astype(float)))

teams = {m["team1"] for m in matches} | {m["team2"] for m in matches}
print("Teams:", len(teams))
print("Missing FIFA points:", [t for t in teams if t not in fifa_points])
print("Missing confederation:", [t for t in teams if t not in CONFED])

## 5. History: recent form and head-to-head
Only matches before the tournament are used, so no World Cup result leaks in.

In [ ]:
hist = pd.read_csv(HISTORY_FILE, encoding="utf-8")
hist["home_team"] = hist["home_team"].replace(HISTORY_NAME_FIX)
hist["away_team"] = hist["away_team"].replace(HISTORY_NAME_FIX)
hist = hist[hist["date"] < KICKOFF]
hist = hist[hist["home_team"].isin(teams) | hist["away_team"].isin(teams)]
hist = hist.dropna(subset=["home_score", "away_score"])

team_games = defaultdict(list)   # team -> (date, goals for, goals against)
pair_games = defaultdict(list)   # (A, B) -> (date, goal difference for A)
for d, h, a, hg, ag in zip(hist["date"], hist["home_team"], hist["away_team"],
                           hist["home_score"].astype(int), hist["away_score"].astype(int)):
    team_games[h].append((d, hg, ag))
    team_games[a].append((d, ag, hg))
    pair_games[(h, a)].append((d, hg - ag))
    pair_games[(a, h)].append((d, ag - hg))
for t in team_games:
    team_games[t].sort(key=lambda g: g[0])  # sorted oldest to newest


def recent_form(team, before):
    # average goal difference over the team's last 10 matches
    games = [g for g in team_games.get(team, []) if g[0] < before][-FORM_WINDOW:]
    return sum(gf - ga for _, gf, ga in games) / len(games) if games else 0.0


def h2h_avg_goal_diff(a, b, before):
    # team a's average goal difference against team b (0 if they never met)
    games = [g for g in pair_games.get((a, b), []) if g[0] < before]
    return sum(diff for _, diff in games) / len(games) if games else 0.0


def h2h_matches_played(a, b, before):
    # how many times a and b have met
    return sum(1 for g in pair_games.get((a, b), []) if g[0] < before)

## 6. Build the dataset
One row per match. `match_id`, `date`, `team1` and `team2` are just labels.

| Column | Meaning |
|---|---|
| `fifa_points_diff` | team1 points minus team2 points |
| `host_team1`, `host_team2` | 1 if that team is a host |
| `same_confederation` | 1 if both teams share a confederation |
| `round_number` | 1 = group stage ... 6 = final |
| `h2h_matches_played` | past meetings of the two teams |
| `h2h_avg_goal_diff` | team1's average goal difference in those meetings |
| `recent_form_diff` | team1's recent form minus team2's |
| `goal_difference` | **target**: team1 goals minus team2 goals |

In [ ]:
matches.sort(key=lambda m: m["date"])
rows = []
for i, m in enumerate(matches, start=1):
    t1, t2, d = m["team1"], m["team2"], m["date"]
    rows.append({
        "match_id": i,
        "date": d,
        "team1": t1,
        "team2": t2,
        "fifa_points_diff": round(fifa_points[t1] - fifa_points[t2], 2),
        "host_team1": int(t1 in HOSTS),
        "host_team2": int(t2 in HOSTS),
        "same_confederation": int(CONFED[t1] == CONFED[t2]),
        "round_number": m["round_number"],
        "h2h_matches_played": h2h_matches_played(t1, t2, KICKOFF),
        "h2h_avg_goal_diff": round(h2h_avg_goal_diff(t1, t2, KICKOFF), 3),
        "recent_form_diff": round(recent_form(t1, KICKOFF) - recent_form(t2, KICKOFF), 3),
        "goal_difference": m["goals1"] - m["goals2"],
    })

df = pd.DataFrame(rows)
df.to_csv("wc2026_2_1_dataset.csv", index=False)
df.head(10)

## 7. Quick checks

In [ ]:
X_COLS = ["fifa_points_diff", "host_team1", "host_team2", "same_confederation",
          "round_number", "h2h_matches_played", "h2h_avg_goal_diff", "recent_form_diff"]
Y_COL = "goal_difference"

assert len(df) == 104                                   # 104 matches
assert len(X_COLS) == 8                                 # 8 explanatory variables
assert df[X_COLS + [Y_COL]].isna().sum().sum() == 0     # nothing missing
print("All checks passed")
df[X_COLS + [Y_COL]].describe().round(2)

## 8. Download the dataset

In [ ]:
try:
    from google.colab import files
    files.download("wc2026_2_1_dataset.csv")
except ImportError:
    print("Saved as wc2026_2_1_dataset.csv")